In [1]:
import pandas as pd
import MeCab
from collections import Counter
from tqdm import tqdm  # 進捗バー表示用
import unidic_lite
import sys
import re


# ============================
# 設定エリア
# ============================
FILE_PATH = 'dataset/narou_dataset.csv'
TARGET_GENRE_ID = None  # 前回の設定に合わせています
# ============================

# ジャンルマップ定義
genres_map = {
    0: '未設定', 101: '異世界（恋愛）', 102: '現実世界（恋愛）',
    201: 'ハイファンタジー', 202: 'ローファンタジー',
    301: '純文学', 302: 'ヒューマンドラマ', 303: '歴史',
    304: '推理', 305: 'ホラー', 306: 'アクション', 307: 'コメディー',
    401: 'VRゲーム', 402: '宇宙', 403: '空想科学', 404: 'パニック',
    9901: '童話', 9902: '詩', 9903: 'エッセイ', 9904: 'リプレイ',
    9999: 'その他', 9801: 'ノンジャンル'
}

In [ ]:
print("データを読み込んでいます...")

# 必要なカラムのみを指定して読み込み（メモリ節約）
use_cols = ['あらすじ', '作品ジャンル']

try:
    # ファイル読み込み
    df = pd.read_csv(FILE_PATH, usecols=use_cols, encoding='utf-8')
    
    # 欠損値（空のあらすじ）を空文字で埋める
    df['あらすじ'] = df['あらすじ'].fillna('')
    
    # ジャンルフィルタリング
    if TARGET_GENRE_ID is not None:
        original_count = len(df)
        df = df[df['作品ジャンル'] == TARGET_GENRE_ID]
        filtered_count = len(df)
        genre_name = genres_map.get(TARGET_GENRE_ID, '不明')
        print(f"抽出完了: ジャンル「{genre_name}」")
        print(f"データ件数: {original_count} -> {filtered_count} 件")
    else:
        print(f"全件対象: {len(df)} 件のデータを分析します。")

except FileNotFoundError:
    print("エラー: ファイルが見つかりません。パスを確認してください。")
except Exception as e:
    print(f"エラーが発生しました: {e}")

データを読み込んでいます...
全件対象: 537829 件のデータを分析します。


In [3]:
# Windowsでの設定ファイル読み込みエラー回避
tagger_cmd = f"-d \"{unidic_lite.DICDIR}\" -r NUL"

try:
    tagger = MeCab.Tagger(tagger_cmd)
except RuntimeError:
    # 万が一の予備策
    with open("mecabrc", "w") as f: pass
    tagger = MeCab.Tagger(f"-d \"{unidic_lite.DICDIR}\" -r ./mecabrc")

# カウンターの準備
word_counter = Counter()

# ストップワード（これらが正しく機能するようになります）
stop_words = {'する', 'いる', 'ある', 'なる', 'ない', 'れる', 'られる', 'こと', 'もの', 'それ', 'これ', 'よう', 'ため', 'いく', 'くる', 'しまう'}

print("形態素解析を実行中...")

if 'df' in locals() and len(df) > 0:
    for synopsis in tqdm(df['あらすじ']):
        if not isinstance(synopsis, str):
            continue

        # 解析実行
        node = tagger.parseToNode(synopsis)
        
        while node:
            features = node.feature.split(',')
            pos = features[0]       # 品詞
            pos_sub = features[1]   # 品詞細分類
            
            # 条件: 名詞・動詞・形容詞
            if pos in ['名詞', '動詞', '形容詞']:
                # 不要な細分類を除外
                if pos_sub not in ['非自立', '接尾', '数', '代名詞']:
                    
                    # ========================================================
                    # 【修正ポイント】 unidic-lite の列に合わせてインデックスを変更
                    # 6: カタカナ読み (例: スル)
                    # 7: 語彙素 (例: 為る) ← 漢字になりすぎることがある
                    # 10: 書字形基本形 (例: する) ← これを使います
                    # ========================================================
                    if len(features) > 10:
                        base_form = features[10] # 書字形基本形 (unidic)
                    elif len(features) > 7:
                        base_form = features[7]  # 語彙素 (バックアップ)
                    else:
                        base_form = node.surface # そのまま
                    
                    # ストップワード判定（base_formに対して行う）
                    if base_form not in stop_words:
                        word_counter[base_form] += 1
            
            node = node.next
    print("集計完了！")
else:
    print("処理対象のデータがありません。")

形態素解析を実行中...


100%|██████████| 537829/537829 [02:54<00:00, 3088.45it/s]

集計完了！


In [4]:
# 上位何件を表示するか
TOP_N = 30

if len(word_counter) > 0:
    print(f"\n=== TOP {TOP_N} ===")
    print(f"対象ジャンル: {genres_map.get(TARGET_GENRE_ID, '全ジャンル')}")
    print("-" * 40)
    print(f"{'順位':<3} \t| {'単語':<15}\t | {'回数':<10}")
    print("-" * 40)

    for rank, (word, count) in enumerate(word_counter.most_common(TOP_N), 1):
        print(f"{rank:<3} \t| {word:<15}\t | {count:<10}")
    print("-" * 40)
else:
    print("集計結果がありません。")


=== TOP 30 ===
対象ジャンル: 全ジャンル
----------------------------------------
順位  	| 単語             	 | 回数        
----------------------------------------
1   	| 世界             	 | 391338    
2   	| いう             	 | 130627    
3   	| 物語             	 | 124510    
4   	| 話              	 | 119260    
5   	| 少女             	 | 114419    
6   	| 年              	 | 105590    
7   	| 主人             	 | 101479    
8   	| 一              	 | 98993     
9   	| 日              	 | 97918     
10  	| 人              	 | 92944     
11  	| 自分             	 | 85083     
12  	| 思う             	 | 82910     
13  	| 魔法             	 | 79880     
14  	| よる             	 | 72511     
15  	| 中              	 | 71466     
16  	| 少年             	 | 71224     
17  	| 時              	 | 69026     
18  	| 高校             	 | 68631     
19  	| 転生             	 | 67519     
20  	| 事              	 | 66135     
21  	| 言う             	 | 62519     
22  	| 知る             	 | 62412     
23  	| 人間             	 | 61987     


In [2]:
# ============================
# 設定エリア（Top30用）
# ============================
OUTPUT_CSV_NAME = 'genre_word_freq_top30.csv' # 出力ファイル名
TOP_N = 30  # 上位何件を出力するか
# ============================

# MeCabの初期化（Windows/Unidic対応版）
tagger_cmd = f"-d \"{unidic_lite.DICDIR}\" -r NUL"
try:
    tagger = MeCab.Tagger(tagger_cmd)
except RuntimeError:
    # 予備策
    with open("mecabrc", "w") as f: pass
    tagger = MeCab.Tagger(f"-d \"{unidic_lite.DICDIR}\" -r ./mecabrc")

# ストップワード定義
stop_words = {
    'する', 'いる', 'ある', 'なる', 'ない', 'れる', 'られる', 
    'こと', 'もの', 'それ', 'これ', 'よう', 'ため', 
    'いく', 'くる', 'しまう', 'の', 'ん', '<URL>' # <URL>も集計から除外したい場合はここに追加
}

def analyze_and_count(df_target, genre_label, genre_id_val):
    """
    データフレームを受け取り、単語頻度をカウントして上位TOP_Nの結果リストを返す関数
    """
    local_counter = Counter()
    
    if len(df_target) == 0:
        return []

    # あらすじごとのループ
    for synopsis in tqdm(df_target['あらすじ'], desc=f"集計中: {genre_label}", leave=False):
        if not isinstance(synopsis, str):
            continue

        # ==========================================================
        # 【追加】 URLを <URL> に置換する処理
        # ==========================================================
        synopsis = re.sub(r'https?://[\w/:%#\$&\?\(\)~\.=\+\-]+', '<URL>', synopsis)

        # 解析実行
        node = tagger.parseToNode(synopsis)
        while node:
            features = node.feature.split(',')
            pos = features[0]
            pos_sub = features[1]

            if pos in ['名詞', '動詞', '形容詞']:
                if pos_sub not in ['非自立', '接尾', '数', '代名詞']:
                    # Unidic対応
                    if len(features) > 10:
                        base_form = features[10]
                    elif len(features) > 7:
                        base_form = features[7]
                    else:
                        base_form = node.surface
                    
                    if base_form not in stop_words:
                        local_counter[base_form] += 1
            node = node.next

    # 結果をリスト形式に整形
    result_rows = []
    for rank, (word, count) in enumerate(local_counter.most_common(TOP_N), 1):
        result_rows.append({
            'ジャンル名': genre_label,
            '単語': word,
            '出現回数': count
        })
    
    return result_rows

# --- メイン処理 ---

print("データセットを読み込んでいます...")
try:
    # 全データを読み込み
    df_all = pd.read_csv(FILE_PATH, usecols=['あらすじ', '作品ジャンル'], encoding='utf-8')
    df_all['あらすじ'] = df_all['あらすじ'].fillna('')
    print(f"データ読み込み完了: 全 {len(df_all)} 件")

    all_results_list = []

    # 1. 全ジャンル (ALL) の集計
    print("\n=== [1/2] 全作品の集計を実行中 ===")
    results_all = analyze_and_count(df_all, "全ジャンル", "ALL")
    all_results_list.extend(results_all)
    print(f"-> 全作品の集計完了")

    # 2. 各ジャンルごとの集計
    print("\n=== [2/2] ジャンル別集計を実行中 ===")
    
    for gid, gname in genres_map.items():
        df_genre = df_all[df_all['作品ジャンル'] == gid]
        
        if len(df_genre) > 0:
            results_genre = analyze_and_count(df_genre, gname, gid)
            all_results_list.extend(results_genre)

    # CSVに出力
    print(f"\nCSVファイル '{OUTPUT_CSV_NAME}' に書き出しています...")
    df_result = pd.DataFrame(all_results_list)
    
    if not df_result.empty:
        df_result = df_result[['ジャンル名', '単語', '出現回数']]
        df_result.to_csv(OUTPUT_CSV_NAME, index=False, encoding='utf-8-sig')
        print("完了しました！")
        print("\n=== 出力結果プレビュー (先頭5行) ===")
        print(df_result.head())
    else:
        print("エラー: 集計結果が空です。")

except FileNotFoundError:
    print("エラー: ファイルが見つかりません。")
except Exception as e:
    import traceback
    traceback.print_exc()
    print(f"予期せぬエラーが発生しました: {e}")

データセットを読み込んでいます...
データ読み込み完了: 全 537829 件

=== [1/2] 全作品の集計を実行中 ===


-> 全作品の集計完了

=== [2/2] ジャンル別集計を実行中 ===



CSVファイル 'genre_word_freq_top30.csv' に書き出しています...
完了しました！

=== 出力結果プレビュー (先頭5行) ===
   ジャンル名  単語    出現回数
0  全ジャンル  世界  391325
1  全ジャンル  いう  130619
2  全ジャンル  物語  124503
3  全ジャンル   話  119236
4  全ジャンル  少女  114419
